<img src="images/logo.png" width="15%" height="15%">

# **LangChain Essentials**

### **5. Prompt Templates**

#### **5.1 Single Message**

Unlike a Python f-string, a prompt template does not need the variable(s) to be initialized before runtime.

In [4]:
from langchain_mistralai import ChatMistralAI
from dotenv import load_dotenv

load_dotenv()

# My API key is in the MISTRAL_API_KEY environment variable
llm = ChatMistralAI(model="mistral-small-latest")

In [5]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("Hello, my name is {name}")
prompt_template

PromptTemplate(input_variables=['name'], input_types={}, partial_variables={}, template='Hello, my name is {name}')

In [6]:
filled_prompt = prompt_template.invoke({"name": "Tom"}) # You can also use a string when there is a single variable: prompt_template.invoke("Tom") 
filled_prompt

StringPromptValue(text='Hello, my name is Tom')

In [7]:
llm.invoke(filled_prompt)

AIMessage(content="Hello, Tom! It's nice to meet you. How can I assist you today? 😊", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 21, 'total_tokens': 43, 'completion_tokens': 22, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019ef553-93c8-7f80-82ac-b2d1145ce52c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 21, 'output_tokens': 22, 'total_tokens': 43})

#### **5.2 Conversation Prompt**

In [8]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate


chat_template = ChatPromptTemplate.from_messages(
    # Don't forget to pass a LIST
    [
        SystemMessagePromptTemplate.from_template("You are a {mood} assistant"),
        HumanMessagePromptTemplate.from_template("Hello, my name is {name}")
    ]
)

filled_chat_template = chat_template.invoke({"mood": "funny", "name": "Tom"})
llm.invoke(filled_chat_template)

AIMessage(content="Hello Tom! 🎉 *adjusts digital party hat*\n\nWhat's shakin', bacon? 🥓 How can I make your day a little more awesome today? Need jokes, advice, or just someone to laugh with? *winks* Let's make this fun! 😄", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 28, 'total_tokens': 90, 'completion_tokens': 62, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019ef556-067d-7161-951f-5a3a3b1c6011-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 62, 'total_tokens': 90})

#### **5.3 Conversation Placeholder**

In [9]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# let's simulate a conversation
conversation = [
    HumanMessage(content="Hello"),
    AIMessage(content="Hello, how can I help you today?")
]

chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage("You are a helpful assistant"),
        MessagesPlaceholder(variable_name="history"),
        HumanMessagePromptTemplate.from_template("{question}")
    ]
)

filled_chat_template = chat_template.invoke(
    {
        "history": conversation,
        "question": "What is a prompt template?"
    }
)

llm.invoke(filled_chat_template)

AIMessage(content='A **prompt template** is a structured framework or predefined format used to generate consistent and effective prompts for AI models, like me. It helps guide the user or system in phrasing inputs to elicit better, more relevant, or actionable responses from the AI.\n\n### **Key Components of a Prompt Template:**\n1. **Context/Role Assignment** – Defines the AI\'s role (e.g., "You are a helpful assistant").\n2. **Task Description** – Clearly states what the AI should do (e.g., "Summarize this text").\n3. **Input Format** – Specifies how information should be provided (e.g., bullet points, paragraphs).\n4. **Output Requirements** – Defines the expected format (e.g., bullet points, JSON, prose).\n5. **Constraints or Guidelines** – Any rules (e.g., "Keep it under 50 words").\n\n### **Example of a Simple Prompt Template:**\n```plaintext\nAct as a [ROLE]. Analyze the following [INPUT TYPE] and provide:\n1. A summary in [X] sentences.\n2. Key points in bullet form.\n3. A re

#### **5.4 Templates & Chains**

In [11]:
conversation = []

chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage("You are a helpful assistant"),
        MessagesPlaceholder(variable_name="history"),
        HumanMessagePromptTemplate.from_template("{question}")
    ]
)

chain = chat_template | llm

while True:
    question = input("Type your question or 'q' to exit:\n")

    if question.lower()=="q":
        break
    
    response = chain.invoke(
        {
            "history": conversation,
            "question": question
        }
    )

    conversation.append(HumanMessage(content=question))
    conversation.append(response)
    
    for msg in conversation[-2:]:
        msg.pretty_print()

================================ Human Message =================================

hello
================================== Ai Message ==================================

Hello! How can I assist you today?
================================ Human Message =================================

my name is tom
================================== Ai Message ==================================

Hi Tom! Nice to meet you. How can I help you today?
================================ Human Message =================================

what's my name?
================================== Ai Message ==================================

Your name is Tom! 😊
